In [ ]:
# ! pip install openai-whisper

'C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Scripts\pip.exe' was blocked by your organization's Device Guard policy.
Contact your support person for more info.


In [6]:
#pip install sounddevice

In [1]:
import whisper

model = whisper.load_model("base")

In [ ]:
#!pip install sounddevice scipy


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\Lenovo\OneDrive\Desktop\voice_to_textproject
['audio.wav', 'recording.wav', 'requirements.txt', 'voice_to_textproject.ipynb', 'voice_venv']


In [3]:
import shutil

print(shutil.which("ffmpeg"))

C:\Users\Lenovo\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-9.0.1-full_build\bin\ffmpeg.EXE


In [ ]:
#pip install pyaudio

   ---------------------------------------- 0.0/164.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/164.1 kB ? eta -:--:--
   -- ------------------------------------- 10.2/164.1 kB ? eta -:--:--
   --------- ----------------------------- 41.0/164.1 kB 388.9 kB/s eta 0:00:01
   ----------------------- -------------- 102.4/164.1 kB 837.8 kB/s eta 0:00:01
   ---------------------------------------  163.8/164.1 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------  163.8/164.1 kB 1.1 MB/s eta 0:00:01
   -------------------------------------- 164.1/164.1 kB 702.4 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import pyaudio
import wave
import whisper
import numpy as np
import time


# =========================================================
# SETTINGS
# =========================================================

FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
CHUNK = 1024

OUTPUT_FILE = "audio.wav"

# Kitne seconds ki continuous silence ke baad recording stop hogi
SILENCE_LIMIT = 4

# Microphone volume threshold
SILENCE_THRESHOLD = 500

# Starting me maximum kitne seconds speech ka wait karega
START_TIMEOUT = 10


# =========================================================
# STEP 1: INITIALIZE MICROPHONE
# =========================================================

print("\n🎤 Initializing microphone...")

audio = pyaudio.PyAudio()

stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    frames_per_buffer=CHUNK
)


# =========================================================
# STEP 2: START RECORDING
# =========================================================

print("\n================================")
print("       🎤 RECORDING")
print("================================")
print("🗣️ Speak now...")
print("⏸️ Recording will automatically stop")
print("   after 4 seconds of silence.")
print("================================\n")


frames = []

start_time = time.time()
last_speech_time = start_time

speech_started = False


# =========================================================
# STEP 3: RECORD AUDIO
# =========================================================

while True:

    # Read microphone data
    data = stream.read(
        CHUNK,
        exception_on_overflow=False
    )

    frames.append(data)

    # -----------------------------------------------------
    # Convert audio data into numpy array
    # -----------------------------------------------------

    audio_data = np.frombuffer(
        data,
        dtype=np.int16
    )

    # -----------------------------------------------------
    # Calculate volume of current audio chunk
    # -----------------------------------------------------

    volume = np.sqrt(
        np.mean(
            audio_data.astype(np.float32) ** 2
        )
    )

    current_time = time.time()


    # =====================================================
    # CHECK WHETHER USER IS SPEAKING
    # =====================================================

    if volume > SILENCE_THRESHOLD:

        speech_started = True

        # Speech detected, so reset silence timer
        last_speech_time = current_time


    # =====================================================
    # AFTER SPEECH STARTS
    # =====================================================

    if speech_started:

        silence_duration = (
            current_time - last_speech_time
        )

        # Stop after 4 seconds continuous silence
        if silence_duration >= SILENCE_LIMIT:

            print("\n⏹️ 4 seconds of silence detected.")
            print("Recording stopped.")

            break


    # =====================================================
    # IF USER HAS NOT STARTED SPEAKING
    # =====================================================

    else:

        waiting_time = (
            current_time - start_time
        )

        if waiting_time >= START_TIMEOUT:

            print("\n⚠️ No speech detected.")
            print("Recording stopped.")

            break


# =========================================================
# STEP 4: CLOSE MICROPHONE
# =========================================================

stream.stop_stream()
stream.close()

sample_width = audio.get_sample_size(FORMAT)

audio.terminate()


# =========================================================
# STEP 5: SAVE AUDIO FILE
# =========================================================

with wave.open(OUTPUT_FILE, "wb") as wf:

    wf.setnchannels(CHANNELS)

    wf.setsampwidth(sample_width)

    wf.setframerate(RATE)

    wf.writeframes(
        b"".join(frames)
    )


print("\n💾 Audio saved as:")
print(OUTPUT_FILE)


# =========================================================
# STEP 6: LOAD WHISPER MODEL
# =========================================================

print("\n================================")
print("       🤖 WHISPER")
print("================================")

print("⏳ Loading Whisper small model...")
print("Please wait...")


model = whisper.load_model("small")


print("✅ Whisper model loaded!")


# =========================================================
# STEP 7: SPEECH TO TEXT
# =========================================================

print("\n📝 Converting speech to text...")
print("Please wait...")


result = model.transcribe(

    OUTPUT_FILE,

    # CPU ke liye
    fp16=False,

    # Language automatically detect karega
    # Isliye language="en" nahi diya hai

    # Better decoding
    beam_size=5,

    # Previous text ka context maintain karega
    condition_on_previous_text=True,

    # Domain-specific words ke liye guidance
    initial_prompt=(
        "This is a Hindi and English mixed voice transcription. "
        "The speaker may talk about Python, "
        "machine learning, "
        "data analytics, "
        "Excel, "
        "SQL, "
        "Power BI, "
        "Tableau, "
        "statistics, "
        "data science and technology."
    ),

    # No-speech detection
    no_speech_threshold=0.6
)


# =========================================================
# STEP 8: GET TRANSCRIBED TEXT
# =========================================================

text = result["text"].strip()


# =========================================================
# STEP 9: DISPLAY FINAL RESULT
# =========================================================

print("\n================================")
print("       📝 TRANSCRIBED TEXT")
print("================================")

print(text)

print("================================")


# =========================================================
# STEP 10: PROJECT COMPLETED
# =========================================================

print("\n✅ Voice-to-Text conversion completed!")
print("🎵 Audio file:", OUTPUT_FILE)
print("📝 Transcription generated successfully.")


🎤 Initializing microphone...

       🎤 RECORDING
🗣️ Speak now...
⏸️ Recording will automatically stop
   after 4 seconds of silence.


⏹️ 4 seconds of silence detected.
Recording stopped.

💾 Audio saved as:
audio.wav

       🤖 WHISPER
⏳ Loading Whisper small model...
Please wait...


100%|███████████████████████████████████████| 461M/461M [01:43<00:00, 4.69MiB/s]


✅ Whisper model loaded!

📝 Converting speech to text...
Please wait...

       📝 TRANSCRIBED TEXT
Power BI is a business intelligence and data visualization tool of Microsoft. It is used to connect, clean, analyze and visualize data. It is used to make interactive reports and dashboards.

✅ Voice-to-Text conversion completed!
🎵 Audio file: audio.wav
📝 Transcription generated successfully.
